# 🔍 PolicyLens — Full RAG Pipeline Notebook

> **Thapar Institute of Engineering & Technology (TIET) — Academic AI Assistant**

This notebook walks through the **complete, end-to-end pipeline** from raw PDF documents to a working Retrieval Augmented Generation (RAG) system.

---

### 📌 Pipeline Overview

```
PDF Documents
     ↓
Text Extraction (PyMuPDF)
     ↓
Cleaning & Filtering (remove noise, headers, footers)
     ↓
Chunking (RecursiveCharacterTextSplitter)
     ↓
Embedding (BAAI/bge-base-en-v1.5 via HuggingFace)
     ↓
Ingestion → Qdrant Vector Store (Cloud)
     ↓
MMR Retrieval + Cross-Encoder Reranking
     ↓
LLM Answer Generation (Groq Llama 3.3 70B)
     ↓
Conversation Memory (Redis / In-Memory)
     ↓
Multimodal: Voice Input (Whisper) + Vision (Llama 4) + TTS
```

---
## 📦 STEP 0 — Install All Dependencies

In [ ]:
# Install all required packages
!pip install -q langchain langchain-groq langchain-huggingface langchain-qdrant langchain-community
!pip install -q qdrant-client sentence-transformers python-dotenv groq gtts
!pip install -q pymupdf pypdf tiktoken transformers torch
!pip install -q redis upstash-redis
!pip install -q nbformat ipywidgets

print('✅ All dependencies installed.')

---
## 🔑 STEP 1 — Load Environment Variables & API Keys

In [ ]:
import os
from dotenv import load_dotenv

# Load from .env file
load_dotenv()

# ─── API Keys ───────────────────────────────────────────────
GROQ_API_KEY   = os.getenv('GROQ_API_KEY')
QDRANT_URL     = os.getenv('QDRANT_URL')
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
REDIS_URL      = os.getenv('REDIS_URL')           # optional
HF_TOKEN       = os.getenv('HF_TOKEN')            # optional fallback

# ─── Config ─────────────────────────────────────────────────
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'tiet_policy_docs')
PDF_FOLDER      = 'data'          # folder where your PDFs live
CHUNK_SIZE      = 800             # characters per chunk
CHUNK_OVERLAP   = 150             # overlap between chunks
TOP_K           = 5               # number of docs to retrieve

# ─── Verification ───────────────────────────────────────────
print('─' * 50)
print(f'GROQ_API_KEY   : {"✅ Loaded" if GROQ_API_KEY else "❌ MISSING"}')
print(f'QDRANT_URL     : {"✅ Loaded" if QDRANT_URL else "❌ MISSING"}')
print(f'QDRANT_API_KEY : {"✅ Loaded" if QDRANT_API_KEY else "❌ MISSING"}')
print(f'REDIS_URL      : {"✅ Loaded" if REDIS_URL else "⚠️ Not set (in-memory fallback)"}')
print(f'HF_TOKEN       : {"✅ Loaded" if HF_TOKEN else "⚠️ Not set (optional)"}')
print('─' * 50)

---
## 📄 STEP 2 — Load & Extract Text from PDFs (PyMuPDF)

In [ ]:
import fitz   # PyMuPDF
import os
from pathlib import Path

def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """
    Extract text from each page of a PDF.
    Returns a list of dicts: {filename, page, text}
    """
    pages = []
    try:
        doc = fitz.open(pdf_path)
        filename = Path(pdf_path).name
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text('text')
            if text.strip():   # skip completely empty pages
                pages.append({
                    'filename': filename,
                    'page': page_num,
                    'text': text
                })
        doc.close()
    except Exception as e:
        print(f'  ⚠️ Error reading {pdf_path}: {e}')
    return pages


# ─── Run Extraction ──────────────────────────────────────────
all_pages = []
pdf_files = list(Path(PDF_FOLDER).glob('*.pdf'))

if not pdf_files:
    print(f'⚠️ No PDFs found in "{PDF_FOLDER}/". Run download_docs.py first!')
else:
    print(f'Found {len(pdf_files)} PDF file(s). Extracting...\n')
    for pdf_path in pdf_files:
        pages = extract_text_from_pdf(str(pdf_path))
        all_pages.extend(pages)
        print(f'  📄 {pdf_path.name:<50} → {len(pages)} pages')

    print(f'\n✅ Total pages extracted: {len(all_pages)}')

In [ ]:
# Preview a sample page
if all_pages:
    sample = all_pages[0]
    print(f'File   : {sample["filename"]}')
    print(f'Page   : {sample["page"]}')
    print(f'Length : {len(sample["text"])} characters')
    print('─' * 50)
    print(sample['text'][:500], '...')

---
## 🧹 STEP 3 — Clean & Filter Text

Raw PDF text often contains noise: repeated headers/footers, excessive whitespace, page numbers, watermarks. We filter those out.

In [ ]:
import re

# ─── Common noise patterns to remove ────────────────────────
NOISE_PATTERNS = [
    r'Page \d+ of \d+',           # page numbers like "Page 1 of 24"
    r'^\d+$',                      # standalone numbers (page nums)
    r'Thapar Institute.*?\n',      # repeated header lines
    r'TIET.*?\n',
    r'\f',                         # form feed characters
    r'\x0c',                       # another form feed
    r'-{3,}',                      # long dashes
    r'={3,}',                      # long equals
]

MIN_PAGE_LENGTH = 80  # skip pages with less than 80 characters (likely blank/header-only)


def clean_text(text: str) -> str:
    """Remove noise, normalize whitespace."""
    for pattern in NOISE_PATTERNS:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE | re.MULTILINE)
    # Collapse multiple blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


# ─── Apply Cleaning ──────────────────────────────────────────
cleaned_pages = []
skipped = 0

for page in all_pages:
    cleaned = clean_text(page['text'])
    if len(cleaned) >= MIN_PAGE_LENGTH:
        cleaned_pages.append({
            'filename': page['filename'],
            'page': page['page'],
            'text': cleaned
        })
    else:
        skipped += 1

print(f'Total pages before filtering : {len(all_pages)}')
print(f'Pages after filtering         : {len(cleaned_pages)}')
print(f'Pages skipped (too short)     : {skipped}')

In [ ]:
# Compare before and after cleaning for a sample page
if all_pages:
    raw  = all_pages[0]['text'][:300]
    clean = cleaned_pages[0]['text'][:300]
    print('─── RAW TEXT ────────────────────────────────────')
    print(repr(raw))
    print('─── CLEANED TEXT ────────────────────────────────')
    print(repr(clean))

---
## ✂️ STEP 4 — Chunk the Text (RecursiveCharacterTextSplitter)

LLMs have limited context windows. We split each page into smaller overlapping chunks so that:
- Each chunk fits inside the embedding model's token limit
- Overlapping chunks prevent information from being cut off at boundaries

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', ' ', ''],  # priority order
    length_function=len
)

all_chunks = []

for page in cleaned_pages:
    # Create LangChain Document objects with metadata
    doc = Document(
        page_content=page['text'],
        metadata={
            'filename': page['filename'],
            'page': page['page'],
            'source': f"{page['filename']} — Page {page['page']}"
        }
    )
    chunks = splitter.split_documents([doc])
    all_chunks.extend(chunks)

print(f'Total pages            : {len(cleaned_pages)}')
print(f'Total chunks created   : {len(all_chunks)}')
print(f'Average chunk size     : {sum(len(c.page_content) for c in all_chunks) // len(all_chunks) if all_chunks else 0} chars')

In [ ]:
# Inspect a sample chunk
if all_chunks:
    sample_chunk = all_chunks[5]
    print('─── CHUNK CONTENT ───────────────────────────────')
    print(sample_chunk.page_content)
    print('─── METADATA ────────────────────────────────────')
    print(sample_chunk.metadata)

---
## 🧬 STEP 5 — Generate Embeddings (BAAI/bge-base-en-v1.5)

We use the **BGE (BAAI General Embedding)** model — a high-quality, open-source dense retrieval model that outperforms many commercial embeddings on academic text.

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

print('Loading BGE embedding model... (first time may take 1–2 min to download)')

embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5',
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}  # normalize for cosine similarity
)

print('✅ Embedding model loaded.')

# Quick test
test_vec = embedding_model.embed_query('What is the fee structure at TIET?')
print(f'Embedding dimension: {len(test_vec)}')  # should be 768 for bge-base

In [ ]:
# BGE recommends a specific query prefix for retrieval tasks
# (improves retrieval quality slightly)

def embed_query_with_prefix(query: str) -> list[float]:
    """Embed a search query with the BGE-recommended prefix."""
    prefixed = 'Represent this sentence for searching relevant passages: ' + query
    return embedding_model.embed_query(prefixed)

# Test
vec = embed_query_with_prefix('hostel room allotment rules')
print(f'Query embedding shape: {len(vec)} dimensions')
print(f'First 5 values: {vec[:5]}')

---
## 📥 STEP 6 — Ingest Chunks into Qdrant (Vector Store)

We push all chunk embeddings into **Qdrant Cloud** — a high-performance vector database.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

# Connect to Qdrant
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Check if the collection already exists
existing_collections = [c.name for c in client.get_collections().collections]
print(f'Existing collections: {existing_collections}')

if COLLECTION_NAME not in existing_collections:
    print(f'Creating collection: {COLLECTION_NAME}')
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=768,              # bge-base embedding dimension
            distance=Distance.COSINE
        )
    )
    print('✅ Collection created.')
else:
    print(f'✅ Collection "{COLLECTION_NAME}" already exists.')

In [ ]:
# Ingest documents in batches
# ⚠️ WARNING: Only run this ONCE (or if you need to re-index)
# Running it again will create duplicate entries!

BATCH_SIZE = 100

if all_chunks:
    print(f'Ingesting {len(all_chunks)} chunks in batches of {BATCH_SIZE}...')
    
    vector_store = QdrantVectorStore.from_documents(
        documents=all_chunks,
        embedding=embedding_model,
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
        collection_name=COLLECTION_NAME
    )
    
    print(f'✅ Ingestion complete! {len(all_chunks)} chunks stored in Qdrant.')
else:
    print('⚠️ No chunks to ingest. Make sure PDFs are in the data/ folder.')

In [ ]:
# If collection already exists, just connect to it (no re-ingestion)
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embedding_model
)

# Check how many vectors are stored
info = client.get_collection(COLLECTION_NAME)
print(f'Vectors in collection: {info.points_count}')

---
## 🔍 STEP 7 — Test Retrieval (MMR Search)

**Maximum Marginal Relevance (MMR)** retrieves documents that are both relevant to the query AND diverse from each other — preventing the same info from appearing in multiple retrieved chunks.

In [ ]:
# Test MMR retrieval
test_query = 'What is the tuition fee for CSE at Thapar?'

retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 5,             # return top 5 chunks
        'lambda_mult': 0.5  # 0=max diversity, 1=max relevance
    }
)

results = retriever.invoke(test_query)

print(f'Query: "{test_query}"')
print(f'Retrieved {len(results)} chunks:\n')

for i, doc in enumerate(results):
    print(f'─── Result {i+1} ─────────────────────────────────────')
    print(f'File : {doc.metadata.get("filename")}')
    print(f'Page : {doc.metadata.get("page")}')
    print(f'Text : {doc.page_content[:200]}...')
    print()

---
## 🏆 STEP 8 — Cross-Encoder Reranking (Optional)

After MMR retrieval, we use a **cross-encoder** to re-score and reorder the results. Cross-encoders jointly encode the query + document for more accurate relevance scoring.

In [ ]:
try:
    from sentence_transformers import CrossEncoder
    
    print('Loading cross-encoder reranker...')
    cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')
    print('✅ Cross-encoder loaded.')
    
    def rerank(query: str, docs: list, top_k: int = 5) -> list:
        """Rerank retrieved docs using cross-encoder scores."""
        pairs = [(query, doc.page_content) for doc in docs]
        scores = cross_encoder.predict(pairs)
        scored = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
        return [doc for _, doc in scored[:top_k]]
    
    # Test reranking
    reranked = rerank(test_query, results)
    print(f'\nAfter reranking top result:')
    print(f'File: {reranked[0].metadata.get("filename")}')
    print(reranked[0].page_content[:300])

except ImportError:
    print('⚠️ sentence-transformers not installed. Skipping reranking.')
    reranked = results
    cross_encoder = None

---
## 🤖 STEP 9 — Initialize Groq LLM (Llama 3.3 70B)

In [ ]:
from langchain_groq import ChatGroq

# Primary LLM — Llama 3.3 70B (fastest, most capable on Groq)
primary_llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.1,         # low temp = factual, less creative
    max_tokens=1024,
    api_key=GROQ_API_KEY
)

# Fallback LLM — Llama 3.1 8B (faster, smaller, for rate-limit situations)
fallback_llm = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0.1,
    max_tokens=1024,
    api_key=GROQ_API_KEY
)

# Quick test
response = primary_llm.invoke('Say "PolicyLens online" and nothing else.')
print('LLM test response:', response.content)

---
## 📝 STEP 10 — Build the System Prompt & RAG Chain (LangChain LCEL)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM_PROMPT = """
You are PolicyLens, a reliable AI assistant for Thapar Institute of Engineering and Technology (TIET), Patiala.

Your job is to answer questions about TIET academic schemes, policies, fees, courses, and official documents.

Rules:
1. Answer ONLY from the provided context (retrieved TIET documents).
2. Always give point-wise, concise answers using bullet points.
3. Include citations like: (Source: filename.pdf, Page X)
4. If the answer is not in the context, say:
   "I could not find this in the available TIET documents."
5. Never invent fees, dates, or rules.
6. For tables, credit lists, or fees — reproduce the exact values.
7. Keep answers structured and scannable, not long paragraphs.
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}')
])

# Format retrieved docs into context string
def format_context(docs) -> str:
    blocks = []
    char_budget = 16000   # max context characters
    total = 0
    for doc in docs:
        source = f"[Source: {doc.metadata.get('filename')}, Page: {doc.metadata.get('page')}]"
        block  = f"{source}\n{doc.page_content}"
        if total + len(block) > char_budget:
            break
        blocks.append(block)
        total += len(block)
    return '\n\n---\n\n'.join(blocks)


# Full RAG chain using LCEL pipe syntax
def retrieve_docs(query: str) -> list:
    docs = retriever.invoke(query)
    if cross_encoder:
        docs = rerank(query, docs)
    return docs


rag_chain = (
    {
        'context'      : lambda x: format_context(retrieve_docs(x['question'])),
        'question'     : lambda x: x['question'],
        'chat_history' : lambda x: x.get('chat_history', [])
    }
    | prompt
    | primary_llm
    | StrOutputParser()
)

print('✅ RAG chain built.')

---
## 💬 STEP 11 — Ask Questions!

In [ ]:
# ✏️ Change this question to test anything!
question = 'What is the tuition fee for BE CSE at Thapar in 2026?'

answer = rag_chain.invoke({'question': question, 'chat_history': []})

print('━' * 60)
print(f'❓ Question: {question}')
print('━' * 60)
print(answer)
print('━' * 60)

In [ ]:
# Try more queries
test_queries = [
    'What are the hostel room allotment rules?',
    'What subjects are in the ECE course scheme for semester 3?',
    'What is the PhD admission eligibility criteria?',
    'How is CGPA calculated at TIET?',
]

for q in test_queries:
    print(f'\n❓ {q}')
    print('─' * 55)
    ans = rag_chain.invoke({'question': q, 'chat_history': []})
    print(ans[:500], '...' if len(ans) > 500 else '')
    print()

---
## 🧠 STEP 12 — Add Conversation Memory (Redis / In-Memory)

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# Try Redis first, fall back to in-memory
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if REDIS_URL:
        try:
            from langchain_community.chat_message_histories import RedisChatMessageHistory
            return RedisChatMessageHistory(
                session_id=session_id,
                url=REDIS_URL,
                key_prefix='tiet_chat:'
            )
        except Exception as e:
            print(f'Redis failed: {e}. Falling back to in-memory.')
    
    # Simple in-memory fallback
    class InMemoryHistory(BaseChatMessageHistory):
        def __init__(self): self.messages = []
        def add_message(self, msg): self.messages.append(msg)
        def clear(self): self.messages = []
    
    if session_id not in _memory_store:
        _memory_store[session_id] = InMemoryHistory()
    return _memory_store[session_id]

_memory_store = {}


# ─── Full conversational ask function ───────────────────────
def ask(question: str, session_id: str = 'notebook_session') -> str:
    history = get_session_history(session_id)
    recent  = history.messages[-6:]  # last 3 turns
    
    answer = rag_chain.invoke({
        'question'     : question,
        'chat_history' : recent
    })
    
    history.add_message(HumanMessage(content=question))
    history.add_message(AIMessage(content=answer))
    return answer


print('✅ Memory setup complete.')

In [ ]:
# Test a multi-turn conversation
SESSION = 'test_session_001'

q1 = 'What is the tuition fee for MCA at TIET?'
a1 = ask(q1, SESSION)
print(f'Q1: {q1}')
print(f'A1: {a1[:400]}\n')

# Follow-up — refers to previous context
q2 = 'Is that the same as the MBA fee?'
a2 = ask(q2, SESSION)
print(f'Q2: {q2}')
print(f'A2: {a2[:400]}')

---
## 🖼️ STEP 13 — Vision Model (Image Input → RAG Query)

In [ ]:
import base64
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

# Llama 4 Scout Vision model
vision_llm = ChatGroq(
    model='meta-llama/llama-4-scout-17b-16e-instruct',
    temperature=0.1,
    max_tokens=512,
    api_key=GROQ_API_KEY
)

def image_to_base64(image_path: str) -> str:
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')


def extract_topic_from_image(image_path: str) -> str:
    """Send image to Vision LLM, extract the topic/question in it."""
    b64 = image_to_base64(image_path)
    
    message = HumanMessage(content=[
        {
            'type': 'text',
            'text': (
                'Look at this image. If it contains an explicit question, extract it exactly. '
                'Otherwise, summarize the topic in 1-2 sentences for use as a search query. '
                'Respond with ONLY the question or topic summary.'
            )
        },
        {
            'type': 'image_url',
            'image_url': {'url': f'data:image/jpeg;base64,{b64}'}
        }
    ])
    
    response = vision_llm.invoke([message])
    return response.content.strip()


def ask_with_image(image_path: str, session_id: str = 'vision_session') -> dict:
    """Extract topic from image, then run full RAG pipeline."""
    extracted_query = extract_topic_from_image(image_path)
    print(f'Extracted query: {extracted_query}')
    answer = ask(extracted_query, session_id)
    return {'extracted_query': extracted_query, 'answer': answer}


# Test: uncomment and provide an image path
# result = ask_with_image('path/to/your/image.jpg')
# print(result['answer'])

print('✅ Vision pipeline ready. Provide an image path to test.')

---
## 🎙️ STEP 14 — Voice Input (Groq Whisper STT)

In [ ]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def transcribe_audio(audio_path: str) -> str:
    """Transcribe an audio file using Groq Whisper Large v3."""
    with open(audio_path, 'rb') as f:
        transcription = groq_client.audio.transcriptions.create(
            file=(os.path.basename(audio_path), f.read()),
            model='whisper-large-v3',
            response_format='text'
        )
    return transcription.strip()


def ask_with_voice(audio_path: str, session_id: str = 'voice_session') -> dict:
    """Transcribe voice → RAG ask → return text + audio answer."""
    question = transcribe_audio(audio_path)
    print(f'Transcribed: "{question}"')
    answer   = ask(question, session_id)
    return {'question': question, 'answer': answer}


# Test: uncomment and provide an audio path
# result = ask_with_voice('path/to/audio.mp3')
# print(result['answer'])

print('✅ Voice input (STT) ready.')

---
## 🔊 STEP 15 — Text-to-Speech Output (gTTS)

In [ ]:
import io
from gtts import gTTS
from IPython.display import Audio, display

def text_to_speech(text: str) -> bytes:
    """Convert text to MP3 audio bytes using gTTS."""
    tts = gTTS(text=text, lang='en', slow=False)
    buffer = io.BytesIO()
    tts.write_to_fp(buffer)
    buffer.seek(0)
    return buffer.read()


# Test TTS — play in notebook
sample_text = 'Welcome to PolicyLens. I am your Thapar Institute academic assistant.'
audio_bytes = text_to_speech(sample_text)

print(f'Generated {len(audio_bytes)} bytes of audio.')
display(Audio(data=audio_bytes, autoplay=False))

---
## 📊 STEP 16 — Collection Stats & Inspection

In [ ]:
# How many vectors are in the database?
collection_info = client.get_collection(COLLECTION_NAME)
print(f'Collection name  : {COLLECTION_NAME}')
print(f'Vectors stored   : {collection_info.points_count}')
print(f'Vector dimension : {collection_info.config.params.vectors.size}')
print(f'Distance metric  : {collection_info.config.params.vectors.distance}')

In [ ]:
# Browse by document name — see which files are indexed
from collections import Counter

# Scroll through stored points to get metadata
scroll_result = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=1000,
    with_payload=True,
    with_vectors=False
)

# Count chunks per file
file_counts = Counter()
for point in scroll_result[0]:
    filename = point.payload.get('metadata', {}).get('filename', 'unknown')
    file_counts[filename] += 1

print(f'\n📚 Documents indexed in Qdrant:')
print('─' * 55)
for fname, count in sorted(file_counts.items()):
    print(f'  {count:>4} chunks  |  {fname}')

print(f'\nTotal: {sum(file_counts.values())} chunks from {len(file_counts)} files')

---
## ✅ Summary — Full Pipeline at a Glance

| Step | Component | Tool |
|------|-----------|------|
| PDF Loading | `fitz.open()` | PyMuPDF |
| Text Cleaning | Regex filtering | Python re |
| Chunking | `RecursiveCharacterTextSplitter` | LangChain |
| Embeddings | `BAAI/bge-base-en-v1.5` | HuggingFace |
| Vector Store | Qdrant Cloud | Qdrant |
| Retrieval | MMR Search | LangChain + Qdrant |
| Reranking | ms-marco MiniLM-L-12 | sentence-transformers |
| LLM | Llama 3.3 70B | Groq |
| Prompt | `ChatPromptTemplate` | LangChain LCEL |
| Memory | Redis / In-Memory | LangChain Community |
| Vision | Llama 4 Scout 17B | Groq |
| STT | Whisper Large v3 | Groq |
| TTS | gTTS | Google TTS |

---

### 🚀 Next Steps
- Run `uvicorn app.main:app --reload` to launch the full web app
- Open `http://127.0.0.1:8000` to use the PolicyLens UI
- Push to GitHub and deploy on Railway / Render / Hugging Face Spaces